In [ ]:
{
  "nbformat": 4,
  "nbformat_minor": 0,
  "metadata": {
    "colab": {
      "provenance": [],
      "gpuType": "T4"
    },
    "kernelspec": {
      "name": "python3",
      "display_name": "Python 3"
    },
    "accelerator": "GPU"
  },
  "cells": [
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "# 01 — Domain-Expert QLoRA Fine-Tuning (Google Colab T4 Pipeline)\n",
        "**Owner:** Member 1 (Team Lead)  \n",
        "**Runtime Requirement:** `Runtime -> Change runtime type -> T4 GPU (15 GB VRAM)`"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "# Cell 1: Verify GPU Allocation & Mount Google Drive\n",
        "import torch\n",
        "assert torch.cuda.is_available(), \"ERROR: No GPU detected! Go to Runtime -> Change runtime type -> T4 GPU.\"\n",
        "!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader\n",
        "\n",
        "from google.colab import drive\n",
        "drive.mount('/content/drive')"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "# Cell 2: Clone or Pull Latest Code from GitHub\n",
        "import os\n",
        "\n",
        "GITHUB_USER = \"https://github.com/Saurav00752\"  # <-- Replace with your GitHub username\n",
        "REPO_NAME = \"domain-expert-qlora\"\n",
        "BRANCH = \"dev\"\n",
        "\n",
        "repo_url = f\"https://github.com/{GITHUB_USER}/{REPO_NAME}.git\"\n",
        "\n",
        "%cd /content\n",
        "if not os.path.exists(REPO_NAME):\n",
        "    !git clone -b {BRANCH} {repo_url}\n",
        "%cd /content/{REPO_NAME}\n",
        "!git checkout {BRANCH}\n",
        "!git pull origin {BRANCH}"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "# Cell 3: Install Pinned QLoRA Dependencies\n",
        "!pip install -q -r requirements.txt"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "# Cell 4: Run Pre-Training Dataset Schema & Leakage Audit (Member 2 Gate)\n",
        "!python data/curation_scripts/validate_splits.py --config training/configs/config.yaml"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "# Cell 5: Execute 4-Bit QLoRA Fine-Tuning\n",
        "# Checkpoints and final adapter automatically save to /content/drive/MyDrive/domain-expert-qlora/\n",
        "!python training/train_qlora.py --config training/configs/config.yaml"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "# Cell 6: Inspect Generated Loss Curves & Backup Logs to Google Drive\n",
        "import shutil\n",
        "from IPython.display import Image, display\n",
        "\n",
        "display(Image(filename=\"training/logs/loss_curves.png\"))\n",
        "\n",
        "drive_log_dir = \"/content/drive/MyDrive/domain-expert-qlora/logs\"\n",
        "os.makedirs(drive_log_dir, exist_ok=True)\n",
        "shutil.copy(\"training/logs/loss_curves.png\", f\"{drive_log_dir}/loss_curves.png\")\n",
        "shutil.copy(\"training/logs/training_metrics.json\", f\"{drive_log_dir}/training_metrics.json\")\n",
        "print(f\"[BACKUP COMPLETE] Logs saved to {drive_log_dir}. Download and commit them to GitHub!\")"
      ]
    }
  ]
}